In [1]:
import os
import faiss
import json
import requests
import numpy as np
from PyPDF2 import PdfReader

from sentence_transformers import SentenceTransformer

In [2]:
 reader = PdfReader("your_pdf_name.pdf")
text = ""
for page in reader.pages:
    text += page.extract_text()

In [3]:
def chunk_text(text, chunk_size=500):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

In [4]:
text = text.replace("\t", " ")
text = text.replace("\n", " ")

In [5]:
chunks = chunk_text(text)

In [6]:
print(chunks[10:13])

['e simply stared at me in amazement and went away without saying a word, so that the spider’s web is comfortably hanging in its place to this day. I only at last this morning realized what was wrong. Aie! Why, they are giving me the slip and making off to their summer villas! Forgive the triviality of the expression, but I am in no mood for fine language . . . for everything that had been in Petersburg had gone or was going away for the holidays; for every respectable gentleman of dignified appea', 'rance who took a cab was at once transformed, in my eyes, into a respectable head of a household who after his daily duties were over, was makinghis way to the bosom of his family, to the summer villa; for all the passers-by had now quite a peculiar air which seemed to say to every one they met: “We are only here for the moment, gentlemen, and in another two hours we shall be going off to the summer villa.” If a window opened after delicate fingers, white as snow, had tapped upon the pane,

In [7]:

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(chunks)

vector = np.array(embeddings).astype("float32")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
faiss.normalize_L2(vector)

dimension = vector.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(vector)

In [9]:
def retrieve(query, k=3):
    query_embedding = embedding_model.encode([query])
    query_vector = np.array(query_embedding).astype("float32")
    faiss.normalize_L2(query_vector)

    scores, indices = index.search(query_vector, k)

    results = [chunks[i] for i in indices[0]]
    return results

In [10]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [11]:
def ask(query):
    results = retrieve(query)
    context = "\n\n".join(results)

    prompt = f"""
    You are a question-answering assistant.

    Use ONLY the context below.
    If answer not found, say "Answer not found in context."

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        },
        data=json.dumps({
            "model": "openai/gpt-3.5-turbo",
            "messages": [
                {"role": "user", "content": prompt}
            ]
        })
    )

    result = response.json()
    return result["choices"][0]["message"]["content"]

In [12]:
print(ask(#write your query.))

KeyError: 'choices'